# 02 - HAR baseline vs. model with attention (walk-forward)

## The target and the two models

Target: $y_t=\ln\mathrm{RV}^{(5)}_t$, log of the std of the 5 daily log returns ending at day $t$. (We take logs because volatility is
positive and right-skewed; the log is closer to Gaussian and is standard in the volatility-forecasting literature.)

**Baseline - HAR-RV** (Heterogeneous AutoRegressive model, Corsi 2009). It captures the fact that traders act on different horizons
(day, week, month), so volatility depends on past volatility at several horizons:
$$y_t=\beta_0+\beta_1\ln\mathrm{RV}^{(5)}_{t-1}+\beta_2\ln\mathrm{RV}^{(22)}_{t-1}+\beta_3\ln|r_{t-1}|+\varepsilon_t$$
(week ~5 days, month ~22 days, plus yesterday's absolute return as a proxy for the daily component). It is a plain OLS regression yet
hard to beat, which is why it is *the* benchmark.

**Attention model** = baseline $+\ \gamma_1\Delta\ln V_{t-1}+\gamma_2\Delta\ln V_{t-2}$ with $V$ the pageviews.

## Why walk-forward

K-fold cross-validation shuffles time: the model would be trained on the future to predict the past. Instead we fit on $[0,t_0)$, predict
the next 21 trading days, extend the training window, refit, repeat. Each prediction uses only information available *then*.
The first 750 days (~3 years) are burn-in.

In [ ]:
import os, sys, warnings
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from src import model_pipeline as mp
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": .3})
START, END = "2016-01-01", "2026-10-31"

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.6))
mt, rf = mp.MIN_TRAIN, mp.REFIT
for i, t0 in enumerate(range(mt, mt + 6 * rf * 8, rf * 8)):
    ax.barh(i, t0, color="tab:blue", alpha=.6)
    ax.barh(i, rf, left=t0, color="tab:red")
ax.set_xlabel("trading day"); ax.set_yticks([])
ax.set_title("Expanding window: blue = training, red = the 21 days predicted (every 8th refit shown)")
plt.show()

## One pair in detail: QQQ / Nvidia

In [ ]:
X, y = mp.build("QQQ", "Nvidia", START, END)
pb = mp.walk_forward(X, y, mp.BASE)
pa = mp.walk_forward(X, y, mp.BASE + mp.ATTN)
yt = y.loc[pb.index]
fig, ax = plt.subplots(figsize=(12, 3.6))
yt.plot(ax=ax, lw=.7, label="actual log RV5", color="k")
pb.plot(ax=ax, lw=.7, label="HAR baseline")
pa.plot(ax=ax, lw=.7, label="HAR + attention", alpha=.8)
ax.legend(); ax.set_title("Out-of-sample forecasts"); plt.show()
print("corr(baseline forecast, attention forecast) =", round(pb.corr(pa), 5), "-> the attention features barely move the forecast")

## Out-of-sample $R^2$

$$R^2_{OOS}=1-\frac{\sum_t (y_t-\hat y^{\,alt}_t)^2}{\sum_t (y_t-\hat y^{\,base}_t)^2}$$
Positive = the alternative model has smaller squared error than the baseline. (Campbell & Thompson 2008 use the same
measure against a historical-mean benchmark.) It can be negative: unlike in-sample $R^2$, adding variables can make out-of-sample
forecasts *worse*, because estimating more parameters adds noise.

In [ ]:
res, dl = mp.compare(y, pb, pa)
r2_vs_mean = 1 - ((yt - pb)**2).sum() / ((yt - yt.expanding().mean().shift(1).bfill())**2).sum()
print(f"HAR baseline R2 vs expanding-mean forecast: {r2_vs_mean:.3f}")
print(f"Attention model, OOS R2 gain over HAR     : {100*res['r2_oos_gain']:.3f} %")

The baseline explains ~76% of the variance because volatility is persistent; attention changes that by tenths of a percent.

## All six pairs

In [ ]:
df = pd.read_csv("results/results.csv")
df[["ticker", "page", "n_oos", "r2_oos_baseline", "r2_oos_gain"]].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(df.ticker + " / " + df.page.str[:8], 100*df.r2_oos_gain, color=np.where(df.r2_oos_gain > 0, "tab:green", "tab:red"))
ax.axhline(0, color="k", lw=.8); ax.set_ylabel("OOS R2 gain (%)"); plt.xticks(rotation=20); plt.show()

**Reading.** Gains are in $[-0.42\%,+0.17\%]$ and four of six are negative. Whether the two positive ones are real or noise is a
*statistical* question, answered in notebook 03.